# Modèles Avancés : Fine-tuning de Transformers (DistilBERT & RoBERTa)

Ce notebook implémente des modèles de type Transformers pour la détection de fake news sur le dataset LIAR. 
Nous testons deux modèles (**DistilBERT** et **RoBERTa**) selon deux configurations d'entrée :
- **Expérience A** : Texte du `statement` uniquement.
- **Expérience B** : Texte + Métadonnées concaténées (sujet, parti, job, locuteur).

L'objectif est de comparer ces modèles contextuels aux baselines linéaires (TF-IDF, GloVe) déjà entraînées.

> **Note sur l'exécution CPU** : L'entraînement des Transformers sur CPU est très long. Un mode `USE_SUBSET` est activé par défaut pour permettre de tester le pipeline.

In [29]:
import os
import pandas as pd
import numpy as np
import torch
from pathlib import Path
from datasets import Dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    TrainingArguments, 
    Trainer, 
    EarlyStoppingCallback
)
from sklearn.metrics import (
    accuracy_score, 
    f1_score, 
    confusion_matrix, 
    classification_report
)
from sklearn.utils.class_weight import compute_class_weight

# Configuration des chemins
DATA_DIR = Path("../data/traitees")
MODELS_DIR = Path("../models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Vérification du GPU / CPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Utilisation de l'appareil : {device}")

# MODE CPU : Réduire la taille du dataset
USE_SUBSET = False  
#SUBSET_SIZE = 500  

Utilisation de l'appareil : cpu


## 1. Chargement et Préparation des données

In [30]:
def load_liar_data(data_dir):
    train_df = pd.read_parquet(data_dir / "liar_train.parquet")
    val_df = pd.read_parquet(data_dir / "liar_val.parquet")
    test_df = pd.read_parquet(data_dir / "liar_test.parquet")
    
    if USE_SUBSET:
        train_df = train_df.sample(min(SUBSET_SIZE, len(train_df)), random_state=42)
        val_df = val_df.sample(min(SUBSET_SIZE//2, len(val_df)), random_state=42)
        test_df = test_df.sample(min(SUBSET_SIZE//2, len(test_df)), random_state=42)
        print(f"Mode SUBSET activé : Entraînement sur {len(train_df)} échantillons.")
    
    return train_df, val_df, test_df

train_df, val_df, test_df = load_liar_data(DATA_DIR)
print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

Train: 10240, Val: 1284, Test: 1267


### 1.1 Construction des deux types d'entrée (Exp A & B)

In [31]:
def prepare_inputs(df):
    df['text_input_A'] = df['statement'].fillna("UNKNOWN")
    
    def concat_meta(row):
        statement = str(row.get('statement', 'UNKNOWN'))
        subject = str(row.get('subject', 'UNKNOWN'))
        party = str(row.get('party', 'UNKNOWN'))
        job = str(row.get('job_title', 'UNKNOWN'))
        speaker = str(row.get('speaker', 'UNKNOWN'))
        return f"[CLAIM] {statement} [SUBJECT] {subject} [PARTY] {party} [JOB] {job} [SPEAKER] {speaker}"

    df['text_input_B'] = df.apply(concat_meta, axis=1)
    return df

train_df = prepare_inputs(train_df)
val_df = prepare_inputs(val_df)
test_df = prepare_inputs(test_df)

print("Exemple d'entrée Exp B :", train_df['text_input_B'].iloc[0])

Exemple d'entrée Exp B : [CLAIM] Says the Annies List political group supports third-trimester abortions on demand. [SUBJECT] abortion [PARTY] republican [JOB] State representative [SPEAKER] dwayne-bohac


## 2. Configuration du Fine-tuning

In [32]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    
    acc = accuracy_score(labels, predictions)
    f1_weighted = f1_score(labels, predictions, average='weighted')
    f1_macro = f1_score(labels, predictions, average='macro')
    
    return {
        'accuracy': acc,
        'f1_weighted': f1_weighted,
        'f1_macro': f1_macro
    }

class_weights = compute_class_weight(
    class_weight='balanced', 
    classes=np.unique(train_df['label_binary']), 
    y=train_df['label_binary']
)
weights = torch.tensor(class_weights, dtype=torch.float).to(device)
print(f"Poids des classes (0=FAKE, 1=REAL) : {class_weights}")

Poids des classes (0=FAKE, 1=REAL) : [1.14081996 0.89012517]


### Custom Trainer pour injecter les class_weights

In [33]:
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        loss_fct = torch.nn.CrossEntropyLoss(weight=weights)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

## 3. Entraînement et Évaluation

Pour éviter les erreurs persistantes de callbacks (`on_train_begin`), nous utilisons une fonction utilitaire pour l'évaluation finale.

In [34]:
def run_evaluation(model, dataset, name):
    print(f"\n--- Évaluation finale : {name} ---")
    # On utilise predict() plutôt que evaluate() pour contourner les bugs de callbacks
    # en créant un Trainer minimaliste sans options d'entraînement
    eval_args = TrainingArguments(
        output_dir="./tmp_eval", 
        use_cpu=(device=='cpu'), 
        report_to="none",
        remove_unused_columns=False
    )
    eval_trainer = Trainer(model=model, args=eval_args)
    
    predictions = eval_trainer.predict(dataset)
    logits = predictions.predictions
    labels = predictions.label_ids
    preds = np.argmax(logits, axis=-1)
    
    acc = accuracy_score(labels, preds)
    f1w = f1_score(labels, preds, average='weighted')
    
    print(f"Accuracy: {acc:.4f}")
    print(f"F1 Weighted: {f1w:.4f}")
    print(classification_report(labels, preds, target_names=['FAKE', 'REAL']))
    
    return {"accuracy": acc, "f1_weighted": f1w}

### 3.1 Expérience 1 : DistilBERT

In [ ]:
MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(examples, col='text_input_A'):
    return tokenizer(examples[col], padding="max_length", truncation=True, max_length=64)

def get_hf_dataset(df, col):
    ds = Dataset.from_pandas(df[['label_binary', col]])
    ds = ds.rename_column('label_binary', 'labels')
    tokenized_ds = ds.map(lambda x: tokenize_fn(x, col), batched=True)
    return tokenized_ds

# --- Exp A (Texte seul) ---
train_ds_A = get_hf_dataset(train_df, 'text_input_A')
val_ds_A = get_hf_dataset(val_df, 'text_input_A')
test_ds_A = get_hf_dataset(test_df, 'text_input_A')

args_A = TrainingArguments(
    output_dir="../models/distilbert_A",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8, 
    per_device_eval_batch_size=8,
    num_train_epochs=1 if USE_SUBSET else 3,
    load_best_model_at_end=True,
    metric_for_best_model="f1_weighted",
    report_to="none",
    use_cpu=(device=='cpu')
)

model_A = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
trainer_A = WeightedTrainer(
    model=model_A, args=args_A, train_dataset=train_ds_A, eval_dataset=val_ds_A,
    compute_metrics=compute_metrics, callbacks=[EarlyStoppingCallback(early_stopping_patience=1)]
)
trainer_A.train()
res_db_A = run_evaluation(model_A, test_ds_A, "DistilBERT Exp A")

# --- Exp B (Texte + Métadonnées) ---
train_ds_B = get_hf_dataset(train_df, 'text_input_B')
val_ds_B = get_hf_dataset(val_df, 'text_input_B')
test_ds_B = get_hf_dataset(test_df, 'text_input_B')

args_B = TrainingArguments(
    output_dir=".../models/distilbert_B",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8, 
    per_device_eval_batch_size=8,
    num_train_epochs=1 if USE_SUBSET else 3,
    load_best_model_at_end=True,
    metric_for_best_model="f1_weighted",
    report_to="none",
    use_cpu=(device=='cpu')
)

model_B = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
trainer_B = WeightedTrainer(
    model=model_B, args=args_B, train_dataset=train_ds_B, eval_dataset=val_ds_B,
    compute_metrics=compute_metrics, callbacks=[EarlyStoppingCallback(early_stopping_patience=1)]
)
trainer_B.train()
res_db_B = run_evaluation(model_B, test_ds_B, "DistilBERT Exp B")

Map:   0%|          | 0/10240 [00:00<?, ? examples/s]

Map:   0%|          | 0/1284 [00:00<?, ? examples/s]

Map:   0%|          | 0/1267 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss


### 3.2 Expérience 2 : RoBERTa

In [ ]:
MODEL_NAME_ROBERTA = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME_ROBERTA)

# --- Exp A (Texte seul) ---
train_ds_A_rob = get_hf_dataset(train_df, 'text_input_A')
val_ds_A_rob = get_hf_dataset(val_df, 'text_input_A')
test_ds_A_rob = get_hf_dataset(test_df, 'text_input_A')

args_A_rob = TrainingArguments(
    output_dir="../models/roberta_A",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8, 
    per_device_eval_batch_size=8,
    num_train_epochs=1 if USE_SUBSET else 3,
    load_best_model_at_end=True,
    metric_for_best_model="f1_weighted",
    report_to="none",
    use_cpu=(device=='cpu')
)

model_A_rob = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME_ROBERTA, num_labels=2)
trainer_A_rob = WeightedTrainer(
    model=model_A_rob, args=args_A_rob, train_dataset=train_ds_A_rob, eval_dataset=val_ds_A_rob,
    compute_metrics=compute_metrics, callbacks=[EarlyStoppingCallback(early_stopping_patience=1)]
)
trainer_A_rob.train()
res_rob_A = run_evaluation(model_A_rob, test_ds_A_rob, "RoBERTa Exp A")

# --- Exp B (Texte + Métadonnées) ---
train_ds_B_rob = get_hf_dataset(train_df, 'text_input_B')
val_ds_B_rob = get_hf_dataset(val_df, 'text_input_B')
test_ds_B_rob = get_hf_dataset(test_df, 'text_input_B')

args_B_rob = TrainingArguments(
    output_dir="../models/roberta_B",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8, 
    per_device_eval_batch_size=8,
    num_train_epochs=1 if USE_SUBSET else 3,
    load_best_model_at_end=True,
    metric_for_best_model="f1_weighted",
    report_to="none",
    use_cpu=(device=='cpu')
)

model_B_rob = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME_ROBERTA, num_labels=2)
trainer_B_rob = WeightedTrainer(
    model=model_B_rob, args=args_B_rob, train_dataset=train_ds_B_rob, eval_dataset=val_ds_B_rob,
    compute_metrics=compute_metrics, callbacks=[EarlyStoppingCallback(early_stopping_patience=1)]
)
trainer_B_rob.train()
res_rob_B = run_evaluation(model_B_rob, test_ds_B_rob, "RoBERTa Exp B")

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

c:\Users\Franck\Documents\grp3_projet3_data\.venv\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Franck\.cache\huggingface\hub\models--roberta-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


--- Évaluation finale : RoBERTa Exp B ---


Accuracy: 0.5440
F1 Weighted: 0.3833
              precision    recall  f1-score   support

        FAKE       0.00      0.00      0.00       114
        REAL       0.54      1.00      0.70       136

    accuracy                           0.54       250
   macro avg       0.27      0.50      0.35       250
weighted avg       0.30      0.54      0.38       250



c:\Users\Franck\Documents\grp3_projet3_data\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Franck\Documents\grp3_projet3_data\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Franck\Documents\grp3_projet3_data\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier,

## 4. Synthèse et Comparaison avec les Baselines

In [ ]:
baselines = [
    {"Modèle": "TF-IDF + LogReg", "Input": "Texte seul", "Acc (test)": 0.632, "F1w (test)": 0.632},
    {"Modèle": "TF-IDF + LinearSVC", "Input": "Texte seul", "Acc (test)": 0.631, "F1w (test)": 0.632},
    {"Modèle": "GloVe + LogReg", "Input": "Texte seul", "Acc (test)": 0.599, "F1w (test)": 0.600},
]

results = baselines.copy()
try:
    results.append({"Modèle": "DistilBERT", "Input": "Texte seul (A)", "Acc (test)": res_db_A['accuracy'], "F1w (test)": res_db_A['f1_weighted']})
    results.append({"Modèle": "DistilBERT", "Input": "Texte + Meta (B)", "Acc (test)": res_db_B['accuracy'], "F1w (test)": res_db_B['f1_weighted']})
    results.append({"Modèle": "RoBERTa", "Input": "Texte seul (A)", "Acc (test)": res_rob_A['accuracy'], "F1w (test)": res_rob_A['f1_weighted']})
    results.append({"Modèle": "RoBERTa", "Input": "Texte + Meta (B)", "Acc (test)": res_rob_B['accuracy'], "F1w (test)": res_rob_B['f1_weighted']})
except:
    pass

df_final = pd.DataFrame(results)
display(df_final)

,Modèle,Input,Acc (test),F1w (test)
0,TF-IDF + LogReg,Texte seul,0.632,0.632000
1,TF-IDF + LinearSVC,Texte seul,0.631,0.632000
2,GloVe + LogReg,Texte seul,0.599,0.600000
3,DistilBERT,Texte seul (A),0.616,0.572800
4,DistilBERT,Texte + Meta (B),0.548,0.423851
5,RoBERTa,Texte seul (A),0.544,0.383337
6,RoBERTa,Texte + Meta (B),0.544,0.383337
